<a href="https://colab.research.google.com/github/Ghost-in-the-RLAIF-GaN-DNA-code-shell/forecast-bot-template/blob/Ghost-in-the-RLAIF-GaN-DNA-code-shell-patch-1/old%20localLlm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install "transformers>=4.42" "accelerate>=0.31" "bitsandbytes>=0.43" \
                "sentencepiece" "einops" "duckduckgo_search>=6.1" \
                "beautifulsoup4" "tqdm" "pandas" "jsonschema" "yfinance" "feedparser" \
                "spacy"

import datetime
import json
import os
import requests
import re
import math
import time
import logging
from typing import Optional, Union, Dict, Generator
from functools import lru_cache

import spacy
from google.colab import userdata # Assuming userdata is used here

logger = logging.getLogger(__name__)

# 🧾 User-Agent: personalized for your fork
USER_AGENT = "forecast-bot/0.2 (https://github.com/Ghost-in-the-RLAIF-GaN-DNA-code-shell/forecast-bot-template)"

# ⏱️ Configurable safety knobs
DEFAULT_TIMEOUT_S = 12
MAX_RETRIES = 2
BACKOFF_BASE_S = 0.6  # exponential backoff base

# Assuming these are needed and fetched via userdata in an earlier cell
METACULUS_TOKEN = userdata.get('METACULUS_TOKEN')
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') # Not used in this block but might be used elsewhere
# PERPLEXITY_API_KEY = userdata.get('PERPLEXITY_API_KEY') # Commenting this out as it's no longer required


# Load the small English language model for spaCy - This requires the model to be downloaded separately
# The download command is now in a separate cell below.
nlp = spacy.load("en_core_web_sm")


def find_number_before_percent(s: str) -> Optional[float]:
    """Finds the first number followed by a percentage sign in a string."""
    match = re.search(r"(\d+(\.\d+)?)%", s)
    if match:
        return float(match.group(1))
    return None


def extract_forecast_tree(text: str) -> dict:
    doc = nlp(text)
    nodes = []
    edges = []

    # Step 1: Extract all numbers and percentages
    for token in doc:
        if token.like_num or re.match(r"\d+(\.\d+)?%", token.text):
            value = float(token.text.strip('%'))
            role = "unknown"

            # Step 2: Tag roles based on nearby words
            window = [w.text.lower() for w in doc[max(0, token.i-3):token.i+4]]
            if "forecast" in window:
                role = "main_forecast"
            elif "baseline" in window:
                role = "baseline"
            elif "higher" in window or "lower" in window or "change" in window:
                role = "delta"
            elif "range" in window or "between" in window:
                role = "range"

            nodes.append({"value": value, "role": role, "token": token.text})

    # Step 3: Build edges based on roles
    forecast = next((n for n in nodes if n["role"] == "main_forecast"), None)
    baseline = next((n for n in nodes if n["role"] == "baseline"), None)
    delta = next((n for n in nodes if n["role"] == "delta"), None)
    ranges = [n for n in nodes if n["role"] == "range"]

    if forecast and baseline and delta:
        edges.append({
            "from": forecast["value"],
            "to": baseline["value"],
            "relation": "higher_than_by",
            "delta": delta["value"]
        })

    if forecast and len(ranges) == 2:
        edges.append({
            "from": forecast["value"],
            "to": [ranges[0]["value"], ranges[1]["value"]],
            "relation": "within_range"
        })

    return {"nodes": nodes, "edges": edges}

# Using the logic from post_comment_minimal for post_question_comment
def post_question_comment(question_id: int, comment_text: str, token: str):
    """Posts a comment to a Metaculus question."""
    url = "https://www.metaculus.com/api2/comments/"
    headers = {"Authorization": f"Token {token}"}
    body = {
        "question": question_id,
        "comment_text": comment_text,
        "submit_type": "N",
        "include_latest_prediction": True
    }
    r = requests.post(url, json=body, headers=headers)
    r.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
    return r.status_code, r.json()


def _coerce_probability_to_percent(x: Union[str, float, int]) -> Optional[float]:
    """
    Accepts values like '72%', 72, 72.0, 0.72 and returns a percent in [0,100].
    Returns None if cannot parse.
    """
    if x is None:
        return None
    if isinstance(x, str):
        s = s.strip()
        if s.endswith("%"):
            s = s[:-1].strip()
        try:
            val = float(s)
        except ValueError:
            return None
    else:
        try:
            val = float(x)
        except (TypeError, ValueError):
            return None

    # Auto-detect 0–1 vs 0–100 scale
    if 0.0 <= val <= 1.0:
        val = val * 100.0
    return val

def _make_session(auth_token: str) -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "Authorization": f"Token {auth_token}",
        "Accept": "application/json",
        "Content-Type": "application/json",
        "User-Agent": USER_AGENT,
    })
    return s

def post_question_prediction(
    question_id: int,
    probability_pct: Union[str, float, int],
    *,
    auth_token: str,
    api_base_url: str = "https://www.metaculus.com/api2",
    dry_run: bool = True,  # ✅ Default True for Colab safety
    timeout_s: int = DEFAULT_TIMEOUT_S,
    max_retries: int = MAX_RETRIES,
) -> dict:
    """
    Post a probability to Metaculus. Automatically:
    - Parses inputs like '72%', 72, 72.0, 0.72 to a percent.
    - Clamps to [1, 99]% to avoid extreme values.
    - Retries on 429/5xx and timeouts with exponential backoff.
    - Returns response JSON (or a minimal dict for dry_run).

    Raises ValueError on invalid inputs, RuntimeError on persistent failure.
    """
    if not isinstance(question_id, int) or question_id <= 0:
        raise ValueError(f"question_id must be a positive int, got {question_id!r}")
    if not auth_token:
        raise ValueError("auth_token is required")

    pct = _coerce_probability_to_percent(probability_pct)
    if pct is None or math.isnan(pct):
        raise ValueError(f"Invalid probability value: {probability_pct!r}")

    pct = max(1.0, min(99.0, float(pct)))  # ✅ Clamp to avoid extremes
    prob = round(pct / 100.0, 6)  # server expects 0–1

    url = f"{api_base_url}/questions/{question_id}/predict/"

    if dry_run:
        logger.info("[DRY_RUN] Would post prediction q=%s value=%.2f%% (%.6f)", question_id, pct, prob)
        return {"dry_run": True, "question_id": question_id, "percent": pct, "probability": prob}

    session = _make_session(auth_token)
    attempt = 0
    last_err = None

    while attempt <= max_retries:
        try:
            resp = session.post(url, json={"prediction": prob}, timeout=timeout_s)
            if resp.ok:
                try:
                    payload = resp.json()
                except ValueError:
                    payload = {"status_code": resp.status_code, "text": resp.text}
                logger.info("Posted prediction: q=%s status=%s percent=%.2f", question_id, resp.status_code, pct)
                return payload

            # Retry on rate-limit/server errors
            if resp.status_code in (429, 500, 502, 503, 504):
                attempt += 1
                retry_after = resp.headers.get("Retry-After")
                sleep_s = float(retry_after) if retry_after and retry_after.isdigit() else BACKOFF_BASE_S * (2 ** (attempt - 1))
                logger.warning("Predict transient error q=%s status=%s attempt=%s/%s; sleeping %.2fs",
                               question_id, resp.status_code, attempt, max_retries, sleep_s)
                time.sleep(sleep_s)
                continue

            # Non-retryable client errors
            snippet = resp.text[:500]
            raise RuntimeError(f"Prediction failed q={question_id} status={resp.status_code} body={snippet}")

        except requests.Timeout as e:
            attempt += 1
            last_err = e
            sleep_s = BACKOFF_BASE_S * (2 ** (attempt - 1))
            logger.warning("Predict timeout q=%s attempt=%s/%s; sleeping %.2fs", question_id, attempt, max_retries, sleep_s)
            time.sleep(sleep_s)
        except requests.RequestException as e:
            attempt += 1
            last_err = e
            sleep_s = BACKOFF_BASE_S * (2 ** (attempt - 1))
            logger.warning("Predict network error q=%s attempt=%s/%s; sleeping %.2fs (%r)",
                           question_id, attempt, max_retries, sleep_s, e)
            time.sleep(sleep_s)

    raise RuntimeError(f"Failed to post prediction for q={question_id} after {max_retries} retries") from last_err


def _retry_get(
    url: str,
    session: requests.Session,
    params: Optional[Dict] = None,
    timeout: float = DEFAULT_TIMEOUT_S,
    max_retries: int = MAX_RETRIES
) -> requests.Response:
    """GET with exponential backoff on 429 and 5xx errors."""
    attempt = 0
    while True:
        resp = session.get(url, params=params, timeout=timeout)
        if resp.ok:
            return resp
        # retryable statuses
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
            backoff = BACKOFF_BASE_S * (2 ** attempt)
            logging.warning(
                "GET retry %s/%s for %s: status=%s; sleeping %.2fs",
                attempt+1, max_retries, url, resp.status_code, backoff
            )
            time.sleep(backoff)
            attempt += 1
            continue
        # non-retryable or exhausted
        snippet = resp.text[:500]
        raise RuntimeError(f"GET {url} failed status={resp.status_code} body={snippet}")

@lru_cache(maxsize=128)
def get_question_details(
    question_id: int,
    auth_token: str,
    api_base_url: str = "https://www.metaculus.com/api2",
) -> Dict:
    """
    Fetch full question JSON. Caches up to 128 unique IDs per session.
    Raises RuntimeError on HTTP errors.
    """
    if question_id <= 0:
        raise ValueError(f"question_id must be positive, got {question_id}")
    url = f"{api_base_url}/questions/{question_id}/"
    session = _make_session(auth_token)
    resp = _retry_get(url, session)
    data = resp.json()

    # accommodate nested new-format: data['question']
    return data.get("question", data)

def list_questions(
    tournament_id: int,
    auth_token: str,
    *,
    offset: int = 0,
    limit: int = 10,
    forecast_type: str = "binary",
    status: str = "open",
    include_description: bool = True,
    api_base_url: str = "https://www.metaculus.com/api2",
) -> Dict:
    """
    Return one page of questions. Raises RuntimeError on HTTP errors.
    To iterate all pages, call in a loop or convert to generator.
    """
    params = {
        "project": tournament_id,
        "offset": offset,
        "limit": limit,
        "forecast_type": forecast_type,
        "status": status,
        "has_group": "false",
        "order_by": "-activity",
        "include_description": str(include_description).lower(),
        "type": "forecast",
    }
    url = f"{api_base_url}/questions/"
    session = _make_session(auth_token)
    resp = _retry_get(url, session, params=params)
    return resp.json()

def iter_all_questions(
    tournament_id: int,
    auth_token: str,
    *,
    page_size: int = 20,
) -> Generator[Dict, None, None]:
    """
    Generator that yields each question dict for a tournament.
    """
    offset = 0
    while True:
        page = list_questions(
            tournament_id, auth_token,
            offset=offset, limit=page_size
        )
        results = page.get("results", [])
        if not results:
            break
        for q in results:
            yield q
        offset += page_size


def duckduckgo_search(query: str, max_results: int = 5) -> list[str]:
    url = f"https://lite.duckduckgo.com/lite/?q={query}"
    headers = {"User-Agent": "forecast-bot/0.2 (+https://github.com/Ghost-in-the-RLAIF-GaN-DNA-code-shell/forecast-bot-template)"}
    r = requests.get(url, headers=headers)
    links = re.findall(r'href="(https?://[^"]+)"', r.text)
    return links[:max_results]

def wikipedia_summary(title: str) -> str:
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title.replace(' ', '_')}"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        return data.get("extract", "")
    return ""

def gdelt_recent_news(query: str) -> str:
    # GDELT RSS search
    rss_url = f"https://blog.gdeltproject.org/feed/?s={query.replace(' ', '+')}"
    r = requests.get(rss_url)
    if r.ok:
        return r.text[:500]  # crude fallback; ideally parse RSS
    return ""


def fetch_finance_brief(query: str) -> str:
    # Placeholder: Detect tickers and fetch from Yahoo Finance or MarketWatch
    return ""

def copilot_search_brief(llm: 'LocalLlm', question_text: str) -> str:
    """
    CopilotSearch orchestrates sources in a forecast-first order:
    1) Wikipedia (background)
    2) GDELT (recency)
    3) DuckDuckGo candidates (+ page excerpts for top links)
    4) Finance layer (if tickers detected)
    Then compresses into a concise brief with a Yes/No lean (no percentages).
    """
    wiki = wikipedia_summary(question_text)
    gdelt = gdelt_recent_news(question_text)
    links = duckduckgo_search(question_text)
    finance = fetch_finance_brief(question_text)

    # Compose raw research block
    raw_context = f"""
Wikipedia:\n{wiki}\n
GDELT:\n{gdelt}\n
DuckDuckGo top links:\n{chr(10).join(links)}\n
Finance:\n{finance}
"""

class LocalLlm:
    def __init__(self, model_id: str = "mistral", temperature: float = 0.2, max_new_tokens: int = 700):
        self.model_id = model_id
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens
        self.active_model_id = model_id

    def __call__(self, prompt: str) -> str:
        import requests
        url = "http://localhost:11434/api/generate"  # Ollama default
        payload = {
            "model": self.model_id,
            "prompt": prompt,
            "temperature": self.temperature,
            "max_tokens": self.max_new_tokens,
            "stream": False
        }
        try:
            r = requests.post(url, json=payload)
            r.raise_for_status()
            return r.json().get("response", "").strip()
        except Exception as e:
            return f"[Local LLM call failed: {e}]"

DEFAULT_PRIMARY_MODEL = "mistral"  # or "llama3", "gemma", etc.

llm = LocalLlm(
    model_id=DEFAULT_PRIMARY_MODEL,
    temperature=0.2,
    max_new_tokens=700
)

def test_llm_connection(llm: LocalLlm) -> bool:
    try:
        response = llm("Say 'ready' if you received this prompt.")
        return "ready" in response.lower()
    except Exception:
        return False

    # Compress with local LLM
    prompt = f"""You are a research assistant helping a forecaster.
Summarize the following multi-source context into a concise brief.
Lean Yes or No if possible, but do not give a probability.

Context:\n{raw_context}
"""
    try:
        brief = llm(prompt)
    except Exception as e:
        brief = f"[LLM compression failed: {e}]"

    return brief.strip()


# Commenting out the copilot_search_brief function definition
# def copilot_search_brief(query: str, api_key: str) -> str:
#     """
#     Calls Perplexity API to get a summary report for a query.
#     """
#     url = "https://api.perplexity.ai/chat/completions"
#     headers = {
#         "accept": "application/json",
#         "authorization": f"Bearer {api_key}",
#         "content-type": "application/json",
#     }
#     payload = {
#         "model": "llama-3.1-sonar-large-128k-chat",
#         "messages": [
#             {
#                 "role": "system",
#                 "content": """
# You are an assistant to a superforecaster.
# The superforecaster will give you a question they intend to forecast on.
# To be a great assistant, you generate a concise but detailed rundown of the most relevant news, including if the question would resolve Yes or No based on current information.
# You do not produce forecasts yourself.
# """,
#             },
#             {"role": "user", "content": query},
#         ],
#     }
#     response = requests.post(url=url, json=payload, headers=headers)

#     if not response.ok:
#         # Log the error and raise an exception
#         logger.error(f"Perplexity API call failed: {response.text}")
#         raise Exception(f"Perplexity API call failed: {response.text}")

#     content = response.json()["choices"][0]["message"]["content"]
#     return content



def get_gpt_prediction(
    llm: 'LocalLlm', # Assuming 'LocalLlm' is defined elsewhere or is a placeholder
    question_details: dict,
    *,
    clamp_min: int = 1,
    clamp_max: int = 99,
    # Removed perplexity_api_key as it's no longer a required argument
) -> tuple[Optional[int], str, str, dict]:
    """
    Generate a forecast using a local LLM.
    Returns:
      (probability_percent, summary_report, full_text, parsed_struct)
    """

    today = datetime.datetime.now().strftime("%Y-%m-%d")

    # Extract necessary fields from question_details
    title = question_details.get("title", "")
    background = question_details.get("description", "") # Assuming 'description' is the background
    resolution_criteria = question_details.get("resolution_criteria", "")
    fine_print = question_details.get("fine_print", "")
    page_url = question_details.get("page_url", "")
    created_time = question_details.get("created_time", "")
    tags = ", ".join(question_details.get("category_tags", []))
    tournament_name = question_details.get("tournament_name", "")
    author = question_details.get("author_username", "")


    # Get summary report (Perplexity call is commented out)
    # Setting a placeholder since Perplexity is not used
    summary_report = "Perplexity search is not enabled in this configuration."


    # Compose prompt
    user = PROMPT_TEMPLATE.format(
        title=title,
        summary_report=summary_report, # This will now contain the placeholder message
        today=today,
        background=background,
        resolution_criteria=resolution_criteria,
        fine_print=fine_print,
        page_url=page_url,
        created_time=created_time,
        category_tags=tags,
        tournament_name=tournament_name,
        # Add other potential placeholders from PROMPT_TEMPLATE if needed
        # Example: question_id=question_details.get("id", "")
    )

    # Call local LLM
    full_text = "" # Initialize full_text
    try:
        # Assuming llm is an object that can be called like a function with the prompt string
        full_text = llm(user)
    except Exception as e:
        logger.error(f"LLM call failed: {e}")
        return None, summary_report, full_text, {"error": f"llm_call_failed: {e}"}

    # Parse JSON block
    parsed_struct = {}
    prob_pct = None
    try:
        start = full_text.find("{")
        end = full_text.rfind("}")
        candidate = full_text[start:end+1] if start != -1 and end != -1 and end > start else full_text
        # Attempt to parse JSON
        parsed_struct = json.loads(candidate)
        raw = parsed_struct.get("probability_percent")
        if raw is not None:
            val = float(str(raw).strip().rstrip("%"))
            if 0.0 <= val <= 1.0:
                val *= 100.0
            prob_pct = int(round(max(clamp_min, min(clamp_max, val))))
    except json.JSONDecodeError:
        # If JSON parsing fails, log it and continue
        logger.warning(f"JSON parsing failed. Raw text: {full_text}")
        parsed_struct = {"raw": full_text, "json_parse_failed": True}
    except Exception as e:
        # Catch other potential errors during JSON processing
        logger.warning(f"Error processing JSON: {e}. Raw text: {full_text}")
        parsed_struct = {"raw": full_text, "json_process_failed": str(e)}


    # Fallback: regex if JSON fails or no probability found in JSON
    if prob_pct is None:
        # find_number_before_percent is now uncommented
        fallback_prob = find_number_before_percent(full_text)
        if fallback_prob is not None:
             prob_pct = int(round(max(clamp_min, min(clamp_max, fallback_prob))))
             logger.info(f"Using regex fallback, extracted probability: {prob_pct}%")
        else:
             logger.warning("Regex fallback failed to find a probability.")


    # Ensure parsed_struct is a dictionary
    if not isinstance(parsed_struct, dict):
         # This case should be less likely after fixing parsing
         parsed_struct = {"raw_output": full_text, "parsing_attempted": True, "final_struct_issue": True}


    return prob_pct, summary_report, full_text, parsed_struct